# QuKerNet完整链条的解析功能演示

接入C08进阶、C09与C11。按“线路生成→核对齐标签→神经评分→筛选→参数更新→下游预测”顺序运行。**本节采用2比特、构造角度和构造分类标签，使用本地精确态计算。它展示接口与数学行为，不是本轮8比特股票结果，不消耗云shots。**

为保持轻环境，神经评分器以NumPy实现128单元ReLU、Smooth L1和Adam；正式股票实验仍使用锁定的研究实现。这里的32个新候选、完整构造训练集梯度等演示配额不覆盖真实协议。真实协议的50股、16/16/50样本与两臂配额保持。公开基础协议默认3小时；本次依据单独授权取消总时限，仅续作原定评价配额，详见[续跑授权说明](../../docs/QuKerNet固定配额续跑说明_20261115.md)。

来源：[L001正式版](https://proceedings.iclr.cc/paper_files/paper/2024/file/4d9944ab3330fe6af8efb9260aa9f307-Paper-Conference.pdf)、[作者固定代码](https://github.com/iamconglei-ctrl/QuKerNet/tree/c9948606145eb85c2bf53f27acebaf5a071a3fa7)。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'课程大纲.md').is_file())
import sys
sys.path.insert(0, str(root/'experiments/qukernet_finance'))
from cloud_v2 import circuits as qc, search
from cloud_v3.numerical_checks import kernel_matrix, exact_pair_shift
from sklearn.svm import SVC
rng = np.random.default_rng(20261115)
x = np.array([[.13,.27],[.42,.63],[.81,.31],[1.07,.94],[1.29,.48],[1.56,1.12],[1.82,.71],[2.06,1.44]])
y = np.array([0,0,1,1,0,1,0,1])
x_eval = np.array([[.25,.39],[.96,.77],[1.44,.82],[1.95,1.26]])
y_eval = np.array([0,1,0,1])
layouts = qc.sample_unique(6, 2, 2, [1,2,3], 20261115)
targets = np.array([search.alignment_and_gradient(kernel_matrix(x,l,[],[]),y)[0] for l in layouts])
assert np.all(np.isfinite(targets))
print('构造线路KTA标签：', targets.round(6))


## 用4条标签训练神经评分器

神经评分器的输入是线路编码，输出是训练KTA的预测值；下游SVC的输入则是核矩阵。它们是两个不同模型。四条训练线路只能做功能演示，剩余两条诊断不能证明神经排序泛化。


In [ ]:
codes = np.stack([qc.encode_layout(l, maximum_depth=3) for l in layouts])
def fit_neural_scorer(a,b,seed=20261115):
    r=np.random.default_rng(seed); d=a.shape[1]
    w=[r.normal(0,np.sqrt(2/d),(d,128)),np.zeros(128),r.normal(0,.03,(128,1)),np.zeros(1)]
    m=[np.zeros_like(v) for v in w]; v=[np.zeros_like(z) for z in w]
    for t in range(1,31):
        hidden=np.maximum(0,a@w[0]+w[1]); predicted=(hidden@w[2]+w[3]).ravel()
        error=predicted-b*10
        delta=(np.clip(error,-1,1)/len(b))[:,None]
        dh=(delta@w[2].T)*(hidden>0)
        g=[a.T@dh,dh.sum(0),hidden.T@delta,delta.sum(0)]
        for j in range(4):
            m[j]=.9*m[j]+.1*g[j];v[j]=.999*v[j]+.001*g[j]*g[j]
            w[j]-=.01*(m[j]/(1-.9**t))/(np.sqrt(v[j]/(1-.999**t))+1e-8)
    return lambda z: ((np.maximum(0,z@w[0]+w[1])@w[2]+w[3]).ravel()/10)
scorer=fit_neural_scorer(codes[:4],targets[:4])
display(pd.DataFrame({'heldout_KTA':targets[4:],'neural_prediction':scorer(codes[4:])}))
new = qc.sample_unique(32,2,2,[1,2,3],20261116,excluded=[qc.layout_code(l) for l in layouts])
scores=scorer(np.stack([qc.encode_layout(l,maximum_depth=3) for l in new]))
chosen=min(range(len(new)),key=lambda i:(-scores[i],qc.digest(new[i])))
layout=new[chosen]
print('新候选神经评分数：',len(new),'；实际选中线路深度：',layout['depth'])


## 保存初值，累加重叠线路两端的参数导数

对每个替换参数，左端与右端的物理门出现都参与参数移位，再用链式法则得到KTA梯度。每步沿梯度上升。层数为1时某些公共参数会在保真度内积中抵消，梯度为零是科学结果，不应自动换线路。


In [ ]:
mask=qc.make_mask(layout,20261115,0,2,exact_parameters=2)
theta0=qc.initial_parameters(x,layout,mask);theta=theta0.copy()
def objective_and_gradient(theta):
    k=kernel_matrix(x,layout,mask,theta)
    dk=np.zeros((len(theta),len(x),len(x)))
    for i in range(len(x)):
        for j in range(i):
            g,_=exact_pair_shift(x[i],x[j],layout,mask,theta)
            dk[:,i,j]=dk[:,j,i]=g
    return search.alignment_and_gradient(k,y,'classification',dk)
value,grad=objective_and_gradient(theta0)
eps=1e-6;fd=[]
for j in range(2):
    delta=np.eye(2)[j]*eps
    plus=search.alignment_and_gradient(kernel_matrix(x,layout,mask,theta0+delta),y)[0]
    minus=search.alignment_and_gradient(kernel_matrix(x,layout,mask,theta0-delta),y)[0]
    fd.append((plus-minus)/(2*eps))
np.testing.assert_allclose(grad,fd,atol=1e-7,rtol=1e-6)
trajectory=[{'step':0,'KTA':value}]
for step in range(1,3):
    _,gradient=objective_and_gradient(theta)
    theta=theta+.2*gradient
    trajectory.append({'step':step,'KTA':objective_and_gradient(theta)[0]})
checkpoint=json.loads(json.dumps({'layout':layout,'mask':mask,'theta0':theta0.tolist(),'theta':theta.tolist()}))
np.testing.assert_allclose(kernel_matrix(x,layout,mask,theta),kernel_matrix(x,checkpoint['layout'],checkpoint['mask'],checkpoint['theta']),atol=0,rtol=0)
display(pd.DataFrame(trajectory))


## 下游预测与解释边界

用相同获选线路、相同替换位置比较终点与自身初值，二者都重新拟合SVC。这不是所有配置的平均微调效应，也不是最优未微调方法比较。构造数据准确率不能回答股票预测有效与否。


In [ ]:
def evaluate(theta):
    all_k=kernel_matrix(np.vstack([x,x_eval]),layout,mask,theta)
    k,cross=all_k[:len(x),:len(x)],all_k[len(x):,:len(x)]
    kp,bp,mapping=search.spectral_classification_map(k,cross)
    pred=SVC(C=1,kernel='precomputed').fit(kp,y).predict(bp)
    return float(np.mean(pred==y_eval)),mapping['correction_frobenius']
before,correction0=evaluate(theta0);after,correction1=evaluate(theta)
display(pd.DataFrame({'checkpoint':['initial','endpoint'],'analytic_accuracy':[before,after],
                      'spectral_correction':[correction0,correction1]}))
print('仅解析功能演示；本节真实数据读取0、云提交0。')


## 真实股票实验入口默认关闭

解析演示不自动转成真实运行。生产入口位于`cloud_v3.runner`；不带`--execute`只返回禁用状态。只有冻结数据、当前账户权限、预算、来源哈希和启动清单全部通过，维护者才在私有环境显式运行。

```text
python -m cloud_v3.runner
python -m cloud_v3.runner --execute --root <PRIVATE_RUN_ROOT> --manifest <PRIVATE_MANIFEST_RELATIVE_PATH> --manifest-sha256 <FROZEN_SHA256>
```

凭据由安全桥在进程输入中注入，不写Notebook、命令参数或输出。下面的开关即使被打开也只提示使用受保护入口，不会提交。


In [ ]:
RUN_REAL_STUDY = False
if RUN_REAL_STUDY:
    raise RuntimeError('请在获授权的私有研究环境使用冻结启动清单；此教学Notebook不提交云任务。')
print('真实运行开关关闭。')


## 作业

把深度固定为1与2，核查共享参数梯度何时抵消；用有限差分验证移位导数；本轮真实先导对256条新候选作经典神经评分，解析演示对32条评分；两者均不等于相同次数的量子核计算。解释预测候选数与实际核评价次数的区别。真实结果只读[结果与预算Notebook](05_真实结果与预算.ipynb)，解析结果不写入实测CSV。
